In [141]:
from collections import deque
import pandas as pd
import os

In [142]:
project_dir = os.path.dirname(os.getcwd())
games = pd.read_csv(os.path.join(project_dir, "data/cleaned_chess.csv"))

In [143]:
games

,Site,Date,White,Black,Result,WhiteElo,BlackElo,ECO
0,Talcahuano CHI,2012-06-02,"Caceres Leal,Jaime","Farfan Ortiz,V",0.5,2137.0,2206.0,D13
1,Talcahuano CHI,2012-06-02,"Caceres Leal,Jaime","Salinas Herrera,P",0.0,2137.0,2409.0,A46
2,Talcahuano CHI,2012-06-02,"Farfan Ortiz,E","Vizama Riveros,C",0.0,2236.0,2256.0,C42
3,Talcahuano CHI,2012-06-02,"Rojas Sepulveda,E","Diaz Vasquez,M",0.5,2315.0,2212.0,A46
4,Talcahuano CHI,2012-06-02,"Estay Torreblanca,O","Alarcon,Rod",1.0,2214.0,2174.0,C17
...,...,...,...,...,...,...,...,...
3255651,Aktobe KAZ,2026-06-08,"Kabinazar,Nurmukhammed","Tarhan,Adar",0.5,2312.0,2482.0,B11
3255652,Aktobe KAZ,2026-06-08,"Zhalmakhanov,Ramazan","Kornienko,Konstantin",0.0,2478.0,2388.0,B28
3255653,Aktobe KAZ,2026-06-08,"Dronavalli,Harika","Degenbaev,Aziz",0.5,2466.0,2320.0,A06
3255654,Aktobe KAZ,2026-06-08,"Rozum,I","Makhnev,Denis",1.0,2455.0,2551.0,A40


Elo 

In [144]:
games["EloDiff"] = games["WhiteElo"] - games["BlackElo"]
games["AvgElo"] = (games["WhiteElo"] + games["BlackElo"]) / 2
games["HigherRatedPlayer"] = games.loc[:,["WhiteElo", "BlackElo"]].idxmax(axis="columns").replace("BlackElo", 0).replace("WhiteElo", 1)
games

,Site,Date,White,Black,Result,WhiteElo,BlackElo,ECO,EloDiff,AvgElo,HigherRatedPlayer
0,Talcahuano CHI,2012-06-02,"Caceres Leal,Jaime","Farfan Ortiz,V",0.5,2137.0,2206.0,D13,-69.0,2171.5,0
1,Talcahuano CHI,2012-06-02,"Caceres Leal,Jaime","Salinas Herrera,P",0.0,2137.0,2409.0,A46,-272.0,2273.0,0
2,Talcahuano CHI,2012-06-02,"Farfan Ortiz,E","Vizama Riveros,C",0.0,2236.0,2256.0,C42,-20.0,2246.0,0
3,Talcahuano CHI,2012-06-02,"Rojas Sepulveda,E","Diaz Vasquez,M",0.5,2315.0,2212.0,A46,103.0,2263.5,1
4,Talcahuano CHI,2012-06-02,"Estay Torreblanca,O","Alarcon,Rod",1.0,2214.0,2174.0,C17,40.0,2194.0,1
...,...,...,...,...,...,...,...,...,...,...,...
3255651,Aktobe KAZ,2026-06-08,"Kabinazar,Nurmukhammed","Tarhan,Adar",0.5,2312.0,2482.0,B11,-170.0,2397.0,0
3255652,Aktobe KAZ,2026-06-08,"Zhalmakhanov,Ramazan","Kornienko,Konstantin",0.0,2478.0,2388.0,B28,90.0,2433.0,1
3255653,Aktobe KAZ,2026-06-08,"Dronavalli,Harika","Degenbaev,Aziz",0.5,2466.0,2320.0,A06,146.0,2393.0,1
3255654,Aktobe KAZ,2026-06-08,"Rozum,I","Makhnev,Denis",1.0,2455.0,2551.0,A40,-96.0,2503.0,0


<h>Player History</h>
<p>New Features:</p>

In [145]:
player_stats = {}

WhiteGames = []
WhiteWin = []
WhiteLoss = []
WhiteDraw = []
BlackGames = []
BlackWin = []
BlackLoss = []
BlackDraw = []

WhiteWinRate = []
WhiteLossRate = []
WhiteDrawRate = []
BlackWinRate = []
BlackLossRate = []
BlackDrawRate = []

WLast10 = []
BLast10 = []

WasWGames = []
WasWWin = []
WasWLoss = []
WasWDraw = []
WasBGames = []
WasBWin = []
WasBLoss = []
WasBDraw = []
BasWGames = []
BasWWin = []
BasWLoss = []
BasWDraw = []
BasBGames = []
BasBWin = []
BasBLoss = []
BasBDraw = []

WasWWinRate = []
WasWLossRate = []
WasWDrawRate = []
WasBWinRate = []
WasBLossRate = []
WasBDrawRate = []
BasWWinRate = []
BasWLossRate = []
BasWDrawRate = []
BasBWinRate = []
BasBLossRate = []
BasBDrawRate = []

Storing the History:

In [146]:
def update_player(player, result, color):
    stats = player_stats.get(player)
    stats["games"] += 1
    if result == 1:
        if color == "white":
            stats["win"] += 1
            stats["whitegames"] += 1
            stats["whitewin"] += 1
            stats["last10"].append(result)
        else:
            stats["loss"] += 1
            stats["blackgames"] += 1
            stats["blackloss"] += 1
            stats["last10"].append(1 - result)
    elif result == 0.5:
        if color == "white":
            stats["draw"] += 1
            stats["whitegames"] += 1
            stats["whitedraw"] += 1
            stats["last10"].append(result)
        else:
            stats["draw"] += 1
            stats["blackgames"] += 1
            stats["blackdraw"] += 1
            stats["last10"].append(1 - result)
    else:
        if color == "white":
            stats["loss"] += 1
            stats["whitegames"] += 1
            stats["whiteloss"] += 1
            stats["last10"].append(result)
        else:
            stats["win"] += 1
            stats["blackgames"] += 1
            stats["blackwin"] += 1
            stats["last10"].append(1 - result)

In [147]:
for row in games.itertuples():
    white = row.White
    black = row.Black
    result = row.Result
    

    if white not in player_stats:
        player_stats[white] = {"games": 0, "win": 0, "loss": 0, "draw": 0, "whitegames": 0, "blackgames": 0, "whitewin": 0, "whiteloss": 0, "whitedraw": 0, "blackwin": 0, "blackloss": 0, "blackdraw": 0, "last10": deque([], maxlen=10)}
    if black not in player_stats:
        player_stats[black] = {"games": 0, "win": 0, "loss": 0, "draw": 0, "whitegames": 0, "blackgames": 0, "whitewin": 0, "whiteloss": 0, "whitedraw": 0, "blackwin": 0, "blackloss": 0, "blackdraw": 0, "last10": deque([], maxlen=10)}


    white_stats = player_stats.get(white)
    black_stats = player_stats.get(black)

    WhiteGames.append(white_stats["games"])
    WhiteWin.append(white_stats["win"])
    WhiteLoss.append(white_stats["loss"])
    WhiteDraw.append(white_stats["draw"])
    BlackGames.append(black_stats["games"])
    BlackWin.append(black_stats["win"])
    BlackLoss.append(black_stats["loss"])
    BlackDraw.append(black_stats["draw"])
    WasWGames.append(white_stats["whitegames"])
    WasWWin.append(white_stats["whitewin"])
    WasWLoss.append(white_stats["whiteloss"])
    WasWDraw.append(white_stats["whitedraw"])
    WasBGames.append(white_stats["blackgames"])
    WasBWin.append(white_stats["blackwin"])
    WasBLoss.append(white_stats["blackloss"])
    WasBDraw.append(white_stats["blackdraw"])
    BasWGames.append(black_stats["whitegames"])
    BasWWin.append(black_stats["whitewin"])
    BasWLoss.append(black_stats["whiteloss"])
    BasWDraw.append(black_stats["whitedraw"])
    BasBGames.append(black_stats["blackgames"])
    BasBWin.append(black_stats["blackwin"])
    BasBLoss.append(black_stats["blackloss"])
    BasBDraw.append(black_stats["blackdraw"])


    if white_stats["games"]:
        WhiteWinRate.append(white_stats["win"] / white_stats["games"])
        WhiteLossRate.append(white_stats["loss"] / white_stats["games"])
        WhiteDrawRate.append(white_stats["draw"] / white_stats["games"])
    else:
        WhiteWinRate.append(0)
        WhiteLossRate.append(0)
        WhiteDrawRate.append(0)
        
    if black_stats["games"]:
        BlackWinRate.append(black_stats["win"] / black_stats["games"])
        BlackLossRate.append(black_stats["loss"] / black_stats["games"])
        BlackDrawRate.append(black_stats["draw"] / black_stats["games"])
    else:
        BlackWinRate.append(0)
        BlackLossRate.append(0)
        BlackDrawRate.append(0)

    if white_stats["whitegames"]:
        WasWWinRate.append(white_stats["whitewin"] / white_stats["whitegames"])
        WasWLossRate.append(white_stats["whiteloss"] / white_stats["whitegames"])
        WasWDrawRate.append(white_stats["whitedraw"] / white_stats["whitegames"])
    else:
        WasWWinRate.append(0)
        WasWLossRate.append(0)
        WasWDrawRate.append(0)

    if white_stats["blackgames"]:
        WasBWinRate.append(white_stats["blackwin"] / white_stats["blackgames"])
        WasBLossRate.append(white_stats["blackloss"] / white_stats["blackgames"])
        WasBDrawRate.append(white_stats["blackdraw"] / white_stats["blackgames"])
    else:
        WasBWinRate.append(0)
        WasBLossRate.append(0)
        WasBDrawRate.append(0)

    if black_stats["whitegames"]:
        BasWWinRate.append(black_stats["whitewin"] / black_stats["whitegames"])
        BasWLossRate.append(black_stats["whiteloss"] / black_stats["whitegames"])
        BasWDrawRate.append(black_stats["whitedraw"] / black_stats["whitegames"])
    else:
        BasWWinRate.append(0)
        BasWLossRate.append(0)
        BasWDrawRate.append(0)

    if black_stats["blackgames"]:
        BasBWinRate.append(black_stats["blackwin"] / black_stats["blackgames"])
        BasBLossRate.append(black_stats["blackloss"] / black_stats["blackgames"])
        BasBDrawRate.append(black_stats["blackdraw"] / black_stats["blackgames"])
    else:
        BasBWinRate.append(0)
        BasBLossRate.append(0)
        BasBDrawRate.append(0)
    
    if white_stats["last10"]:
        WLast10.append(sum(white_stats["last10"]) / len(white_stats["last10"]))
    else:
        WLast10.append(0)
    if black_stats["last10"]:
        BLast10.append(sum(black_stats["last10"]) / len(black_stats["last10"]))
    else:
        BLast10.append(0)
    

    ###updating the features
    update_player(white, result, "white")
    update_player(black, result, "black")

    
    print(white_stats)
    print(black_stats)
    print(row)
    break

{'games': 1, 'win': 0, 'loss': 0, 'draw': 1, 'whitegames': 1, 'blackgames': 0, 'whitewin': 0, 'whiteloss': 0, 'whitedraw': 1, 'blackwin': 0, 'blackloss': 0, 'blackdraw': 0, 'last10': deque([0.5], maxlen=10)}
{'games': 1, 'win': 0, 'loss': 0, 'draw': 1, 'whitegames': 0, 'blackgames': 1, 'whitewin': 0, 'whiteloss': 0, 'whitedraw': 0, 'blackwin': 0, 'blackloss': 0, 'blackdraw': 1, 'last10': deque([0.5], maxlen=10)}
Pandas(Index=0, Site='Talcahuano CHI', Date='2012-06-02', White='Caceres Leal,Jaime', Black='Farfan Ortiz,V', Result=0.5, WhiteElo=2137.0, BlackElo=2206.0, ECO='D13', EloDiff=-69.0, AvgElo=2171.5, HigherRatedPlayer=0)


Head2Head Statistics

New Features:

In [148]:
head2head = {}

h2h_games = []
h2h_Player1Win = []
h2h_Player2Win = []
h2h_Draw = []
h2h_Player1WinRate = []
h2h_Player2WinRate = []
h2h_Drawrate = []
h2h_last5 = []

Keeping track of H2H:

In [149]:
def update_h2h(pair, result, white, black):
    stats = head2head.get(pair)
    stats["games"] += 1
    stats["last5"].append(result)

    if result == 1:
        stats[white] += 1
    elif result == 0.5:
        stats["draw"] += 1
    else:
        stats[black] += 1

In [150]:
for row in games.itertuples():
    white = row.White
    black = row.Black
    result = row.Result
    pair = tuple(sorted((white, black)))

    if pair not in head2head:
        head2head[pair] = {"games": 0, pair[0]: 0, pair[1]: 0, "draw": 0, "last5": deque([], maxlen=5)}
    
    pair_stats = head2head[pair]

    h2h_games.append(pair_stats["games"])
    h2h_Player1Win.append(pair_stats[pair[0]])
    h2h_Player2Win.append(pair_stats[pair[1]])
    h2h_Draw.append(pair_stats["draw"])
    
    if pair_stats["games"]:
        h2h_Player1WinRate.append(pair_stats[pair[0]] / pair_stats["games"])
        h2h_Player2WinRate.append(pair_stats[pair[1]] / pair_stats["games"])
        h2h_Drawrate.append(pair_stats["draw"] / pair_stats["games"])
        h2h_last5.append(sum(pair_stats["last5"]) / len(pair_stats["last5"]))
    else:
        h2h_Player1WinRate.append(0)
        h2h_Player2WinRate.append(0)
        h2h_Drawrate.append(0)
        h2h_last5.append(0)

    ###updating the h2h features
    update_h2h(pair, result, white, black)

    print(head2head)
    break

{('Caceres Leal,Jaime', 'Farfan Ortiz,V'): {'games': 1, 'Caceres Leal,Jaime': 0, 'Farfan Ortiz,V': 0, 'draw': 1, 'last5': deque([0.5], maxlen=5)}}


Diff 

New Features:

In [151]:
Last10Diff = []
GamesDiff = []
WinDiff = []
WinRateDiff = []
h2h_WinDiff = []
h2h_WinRateDiff = []

Calculating the Diff stats:

In [152]:
for i in range(len(games)):
    Last10Diff.append(WLast10[i] - BLast10[i])
    GamesDiff.append(WhiteGames[i] - BlackGames[i])
    WinDiff.append(WhiteWin[i] - BlackWin[i])
    WinRateDiff.append(WhiteWinRate[i] - BlackWinRate[i])
    h2h_WinDiff.append(h2h_Player1Win[i] - h2h_Player2Win[i])
    h2h_WinRateDiff.append(h2h_Player1WinRate[i] - h2h_Player2WinRate[i])
    print(h2h_WinDiff)
    break

[0]


ECO Stats(opening stats)

New Features:

In [153]:
ECOs = {}

ECO_Games = []
ECO_WhiteWin = []
ECO_BlackWin = []
ECO_Draw = []
ECO_WhiteWinRate = []
ECO_BlackWinRate = []
ECO_DrawRate = []


In [154]:
def update_eco(eco, result):
    stats = ECOs[eco]
    stats["games"] += 1
    if result == 1:
        stats["whitewin"] += 1
    elif result == 0.5:
        stats["draw"] += 1
    else:
        stats["blackwin"] += 1

In [155]:
count = 0
for row in games.itertuples():
    result = row.Result
    eco = row.ECO

    if eco not in ECOs:
        ECOs[eco] = {"games": 0, "whitewin": 0, "blackwin": 0, "draw": 0}
    eco_stats = ECOs[eco]
    
    ECO_Games.append(eco_stats["games"])
    ECO_WhiteWin.append(eco_stats["whitewin"])
    ECO_BlackWin.append(eco_stats["blackwin"])
    ECO_Draw.append(eco_stats["draw"])
    if eco_stats["games"]:
        ECO_WhiteWinRate.append(eco_stats["whitewin"] / eco_stats["games"])
        ECO_BlackWinRate.append(eco_stats["blackwin"] / eco_stats["games"])
        ECO_DrawRate.append(eco_stats["draw"] / eco_stats["games"])
    else:
        ECO_WhiteWinRate.append(0)
        ECO_BlackWinRate.append(0)
        ECO_DrawRate.append(0)
    
    ###updating features
    update_eco(eco, result)

    if count>1000:   
        print(ECOs)
        break
    count+=1

{'D13': {'games': 4, 'whitewin': 1, 'blackwin': 0, 'draw': 3}, 'A46': {'games': 11, 'whitewin': 4, 'blackwin': 3, 'draw': 4}, 'C42': {'games': 4, 'whitewin': 1, 'blackwin': 2, 'draw': 1}, 'C17': {'games': 4, 'whitewin': 2, 'blackwin': 2, 'draw': 0}, 'D45': {'games': 11, 'whitewin': 2, 'blackwin': 3, 'draw': 6}, 'E61': {'games': 1, 'whitewin': 1, 'blackwin': 0, 'draw': 0}, 'A57': {'games': 3, 'whitewin': 2, 'blackwin': 0, 'draw': 1}, 'B30': {'games': 16, 'whitewin': 3, 'blackwin': 6, 'draw': 7}, 'D79': {'games': 2, 'whitewin': 0, 'blackwin': 1, 'draw': 1}, 'B32': {'games': 2, 'whitewin': 0, 'blackwin': 1, 'draw': 1}, 'A56': {'games': 6, 'whitewin': 5, 'blackwin': 0, 'draw': 1}, 'B44': {'games': 2, 'whitewin': 2, 'blackwin': 0, 'draw': 0}, 'B45': {'games': 2, 'whitewin': 0, 'blackwin': 1, 'draw': 1}, 'B12': {'games': 19, 'whitewin': 7, 'blackwin': 5, 'draw': 7}, 'C41': {'games': 5, 'whitewin': 2, 'blackwin': 2, 'draw': 1}, 'A05': {'games': 5, 'whitewin': 3, 'blackwin': 0, 'draw': 2}, 'D3